# LLM Brain — MWAP Project

## What this is
This notebook is the **reasoning layer** for the MWAP project. It loads every file in the `/MWAP` folder (code, Excel data, PPTX, notebooks) into GPT-6 Astra's context and exposes `ask_brain()` for querying.

## For Genie Code / any coding agent:

### Setup (once per session)
Run **Cell 2** (pip install) and **Cell 3** (`ask_brain()` definition) before anything else. If `ask_brain` is already defined, skip this.

### To consult the brain:
Add a **new cell at the bottom** and call:
```python
answer = ask_brain("<your question — be specific, ask for exact code and file paths>")
```
Run that cell, read the full response, then **execute exactly what the Brain says** — edit files, apply diffs, run code.

### Follow-ups (keeps previous Q&A in memory):
```python
answer = ask_brain("<follow-up question>", follow_up=True)
```

### Reset conversation:
```python
clear_history()
```

## Key behaviors:
- **Auto-refreshes** — re-crawls the entire `/MWAP` folder on every call, so new files and code changes are always included
- **Persistent memory** — conversation history survives kernel restarts (saved to `.brain_history.json`)
- **Reads everything** — `.py`, `.sql`, `.md`, `.yaml`, `.json`, `.xlsx` (all sheets as CSV), `.pptx` (slide text), `.ipynb` (cell extraction)

## What the Brain knows:
- Every file in `/MWAP` and all subdirectories
- All Excel data (column names, values, row counts)
- All code, function signatures, imports, class definitions
- Full project architecture

## What the Brain does NOT know:
- Live Unity Catalog table contents (only static files on disk)
- Running job/pipeline state
- External APIs
- Files outside `/MWAP`

## Endpoint: `databricks-gpt-6-astra` (configurable in Cell 3)

In [0]:
%pip install openai openpyxl python-pptx -q

In [0]:
import os, json, time, openai, pandas as pd
from pptx import Presentation

# ── CONFIG ─────────────────────────────────────────────────────
ENDPOINT = "databricks-gpt-6-1-sol"
PROJECT_ROOT = "/Workspace/Users/tengkuanas.zainalabi@petronas.com.my/MWAP"
SKIP_DIRS = {".git", "__pycache__", "node_modules", ".venv", "outputs"}
SKIP_EXTS = {".xlsx", ".xls", ".pptx"}  # Too large for LLM context — use DATA_DOCUMENTATION.md instead
CODE_EXTS = {
    ".py", ".sql", ".md", ".yaml", ".yml", ".json",
    ".html", ".css", ".js", ".toml", ".sh", ".txt", ".ipynb",
}
MAX_CHARS_PER_FILE = 300_000

SYSTEM_PROMPT = """
You are the Brain — an expert data scientist, ML engineer, and research advisor embedded in a Databricks project.

You have the ENTIRE project loaded in context below: all code, notebooks, configs, AND all data files (Excel sheets converted to CSV).

Your outputs are executed by a coding agent (Genie Code). Write accordingly:

## How to respond:
1. **Be specific** — reference exact file paths, column names, sheet names, function names, line numbers
2. **Write complete code** — no pseudocode, no "implement this part". Every code block must be copy-pasteable and runnable
3. **Show exact diffs** when modifying existing files — format as: FILE: <path>, OLD: <exact text>, NEW: <replacement>
4. **Cite the data** — when proposing approaches, quote actual column names, row counts, value ranges you can see in the loaded context
5. **Think step-by-step** — for complex tasks, break into numbered steps the executor can follow sequentially
6. **Flag unknowns** — if you need live data (table queries, API calls) that isn't in the loaded files, say so explicitly

## What you know:
- Every file in the project directory (re-crawled fresh before each call)
- All Excel sheet contents as CSV (column names, data types, row counts, actual values)
- All code, notebooks, configs, markdown docs
- All PowerPoint slide contents

## What you do NOT know:
- Live database/table contents (only static files)
- Running job or pipeline state
- External API responses
- Files outside the project root
"""

# ── OpenAI client ──────────────────────────────────────────────
_base = spark.conf.get("spark.databricks.workspaceUrl")
if not _base.startswith("http"):
    _base = f"https://{_base}"

_client = openai.OpenAI(
    api_key=dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get(),
    base_url=f"{_base.rstrip('/')}/serving-endpoints",
)

_conversation_history = []
_HISTORY_FILE = os.path.join(PROJECT_ROOT, ".brain_history.json")

def _load_history():
    """Load conversation history from disk (survives kernel restart)."""
    global _conversation_history
    if os.path.exists(_HISTORY_FILE):
        try:
            with open(_HISTORY_FILE, "r") as f:
                _conversation_history = json.load(f)
            print(f"📜 Loaded {len(_conversation_history) // 2} previous exchanges from disk")
        except:
            _conversation_history = []

def _save_history():
    """Persist conversation history to disk."""
    try:
        with open(_HISTORY_FILE, "w") as f:
            json.dump(_conversation_history, f)
    except:
        pass

def clear_history():
    """Reset conversation memory (both in-memory and on disk)."""
    global _conversation_history
    _conversation_history = []
    if os.path.exists(_HISTORY_FILE):
        os.remove(_HISTORY_FILE)
    print("🗑️ Conversation history cleared")

_load_history()

# ── Crawl functions ────────────────────────────────────────────
def _read_excel(fpath):
    """Convert all sheets in an Excel file to text."""
    parts = []
    try:
        xls = pd.ExcelFile(fpath, engine="openpyxl")
        for sheet in xls.sheet_names:
            df = pd.read_excel(xls, sheet_name=sheet)
            parts.append(f"--- SHEET: {sheet} ({df.shape[0]} rows x {df.shape[1]} cols) ---")
            parts.append(df.to_csv(index=False))
    except Exception as e:
        parts.append(f"ERROR reading Excel: {e}")
    return "\n".join(parts)

def _read_pptx(fpath):
    """Extract text from PowerPoint slides."""
    parts = []
    try:
        prs = Presentation(fpath)
        for i, slide in enumerate(prs.slides, 1):
            texts = []
            for shape in slide.shapes:
                if hasattr(shape, "text") and shape.text.strip():
                    texts.append(shape.text)
            if texts:
                parts.append(f"--- SLIDE {i} ---")
                parts.extend(texts)
    except Exception as e:
        parts.append(f"ERROR reading PPTX: {e}")
    return "\n".join(parts)

def _read_notebook(fpath):
    """Extract code/markdown cells from .ipynb."""
    try:
        with open(fpath, "r", encoding="utf-8", errors="ignore") as f:
            nb = json.loads(f.read())
        parts = []
        for cell in nb.get("cells", []):
            src = "".join(cell.get("source", []))
            if src.strip():
                parts.append(f"# [{cell.get('cell_type', '')}]\n{src}")
        return "\n\n".join(parts)
    except:
        return None

def _crawl():
    """Walk the project, read everything, return context blob."""
    files = {}
    for dirpath, dirnames, filenames in os.walk(PROJECT_ROOT):
        dirnames[:] = sorted([d for d in dirnames if d not in SKIP_DIRS])
        for fname in sorted(filenames):
            fpath = os.path.join(dirpath, fname)
            rel = os.path.relpath(fpath, PROJECT_ROOT)
            ext = os.path.splitext(fname)[1].lower()

            content = None
            if ext in SKIP_EXTS:
                continue  # skip — summarized in DATA_DOCUMENTATION.md
            elif ext == ".pptx":
                content = _read_pptx(fpath)
            elif ext == ".ipynb":
                content = _read_notebook(fpath)
            elif ext in CODE_EXTS:
                try:
                    with open(fpath, "r", encoding="utf-8", errors="ignore") as f:
                        content = f.read()
                except:
                    pass
            elif ext in (".parquet", ".png", ".jpg", ".gif", ".ico", ".woff", ".mp4"):
                continue  # skip binary
            else:
                try:
                    with open(fpath, "r", encoding="utf-8", errors="ignore") as f:
                        content = f.read()
                except:
                    continue

            if content is None:
                continue
            if len(content) > MAX_CHARS_PER_FILE:
                content = content[:MAX_CHARS_PER_FILE] + f"\n[TRUNCATED at {MAX_CHARS_PER_FILE:,} chars]"
            files[rel] = content

    # Build blob
    parts = ["FILE TREE", "=" * 50]
    for p in sorted(files):
        parts.append(f"  {p}  ({len(files[p]):,} chars)")
    parts.append("\nFILE CONTENTS\n" + "=" * 50)
    for p in sorted(files):
        parts.append(f"\n{'=' * 50}\nFILE: {p}\n{'=' * 50}")
        parts.append(files[p])
    blob = "\n".join(parts)
    return blob, len(files)

# ── ask_brain ──────────────────────────────────────────────────
def ask_brain(question: str, *, follow_up: bool = False, max_tokens: int = 16_000) -> str:
    """
    Ask the Brain. Auto-refreshes context on every call so new files
    are always included.
    """
    global _conversation_history

    # Always re-crawl to pick up new files
    blob, n_files = _crawl()
    est_tok = len(blob) // 4
    print(f"📂 {n_files} files | ~{est_tok:,} tok context | {ENDPOINT}")

    sys_msg = {"role": "system", "content": SYSTEM_PROMPT + "\n\n<PROJECT>\n" + blob + "\n</PROJECT>"}

    if follow_up and _conversation_history:
        messages = [sys_msg] + _conversation_history + [{"role": "user", "content": question}]
    else:
        _conversation_history = []
        messages = [sys_msg, {"role": "user", "content": question}]

    print(f"💬 {question.strip()[:300]}")
    start = time.time()

    resp = _client.chat.completions.create(
        model=ENDPOINT,
        messages=messages,
        max_tokens=max_tokens,
    )

    answer = resp.choices[0].message.content
    elapsed = time.time() - start

    _conversation_history.append({"role": "user", "content": question})
    _conversation_history.append({"role": "assistant", "content": answer})
    _save_history()

    print(f"\n✅ {elapsed:.1f}s | {resp.usage.prompt_tokens:,}+{resp.usage.completion_tokens:,}={resp.usage.total_tokens:,} tok")
    print("=" * 60)
    print(answer)
    return answer

print("✅ ask_brain() ready")
print("   • Auto-refreshes files on every call")
print("   • Conversation history persists to disk (.brain_history.json)")
print("   • ask_brain('question')           → fresh conversation")
print("   • ask_brain('question', follow_up=True) → keeps history")
print("   • clear_history()                  → reset memory")

In [0]:
answer = ask_brain("""
Analyze the entire dataset. What exactly is the problem we're solving?
What are the inputs, outputs, constraints, and evaluation criteria?
What ML approaches would you recommend and why?
""")

In [0]:
answer = ask_brain("""
I need you to produce a COMPLETE, DETAILED dataset documentation in Markdown format. This will be saved as a standalone reference file.

Cover ALL of the following:

## 1. Problem Statement
- What exactly is the ML problem? (history matching + production forecasting)
- What is a reservoir simulation model?
- What are the 4 uncertainty parameters and what do they physically represent?
- What is "history matching" in reservoir engineering context?

## 2. Dataset Inventory
For EVERY Excel file (01 through 09), document:
- Every sheet name, row count, column count
- Every column name with its unit, data type, and value range (min/max/mean)
- What the -999.25 sentinel values mean
- The relationship between files (how they connect)

## 3. Data Architecture
- How the 100 cases are split (train/validation/test)
- The 3D grid structure (I, J, K indices — what are the dimensions?)
- Time axis: history period vs forecast period, quarterly vs daily resolution
- How uncertainty parameters map to grid properties map to production curves

## 4. Input Features Deep Dive
- Static properties: what each one means physically (PORV, PORO, PERMX/Y/Z, NTG, TRANX/Y/Z, etc.)
- Dynamic properties: PRESSURE, SWAT, SOIL, SGAS, RS, RV — what they represent
- The 4 uncertainty multipliers: exact ranges and distributions from the data
- Grid cell statistics: how many active vs inactive cells per case

## 5. Target Variables
- What exactly must we predict? (cumulative oil, gas, water production)
- Template structure: date ranges per case, number of forecast timesteps
- Units for each target

## 6. Key Data Quality Notes
- Sentinel values (-999.25) and how to handle them
- Any missing data patterns
- Date format inconsistencies
- The 02 Field Production time series structure (2-row header, Case_1..Case_100 + RAMP_History)

## 7. Suggested ML Approach Summary
- Based on the actual data structure you can see, what architectures make sense?
- What should be the feature engineering strategy?
- How to handle the grid-to-timeseries mapping?

Be EXHAUSTIVE. Quote actual numbers from the data. This is the team's primary reference document.
""")

In [0]:
answer = ask_brain("""
I've completed notebook 01 Data Preparation. It produced 3 parquet files in /MWAP/data/:

1. uncertainty_params.parquet — 100 rows × 7 cols (case_num, case_id, split, Fault Transmissibility, Porosity Multiplier, Permeability Multiplier, Aquifer Pore Volume)
2. production_timeseries.parquet — 6,724,340 rows × 6 cols (date, value, metric, unit, case_num, split) — long format, 10 metrics: oil_cum, oil_rate, gas_cum, gas_rate, water_cum, water_rate + quarterly observed history
3. grid_properties.parquet — 4,251,200 rows × 25 cols (42,512 active cells × 100 cases) — I/J/K indices, XYZ coords, PORV, PORO, PERMX, PERMZ, TRANX/Y/Z, FIPNUM, PVTNUM, SWL, SWCR, SGL, SGCR, SOWCR, PRESSURE, SWAT, SOIL, case_num, split

Now I need to build notebook 02 EDA. Design a COMPLETE EDA notebook with these goals:
- Understand the data deeply before modeling
- Find patterns that inform feature engineering and model architecture
- Create visualizations that tell the story for the PowerPoint deliverable

For each cell, give me:
1. A descriptive cell title
2. Complete, runnable Python code (using matplotlib/seaborn, loading from parquet)
3. What insight the cell reveals

Cover AT LEAST:
- Uncertainty parameter distributions and correlations
- Production curve families (all 100 cases overlaid, color by split)
- How uncertainty params correlate with final cumulative production
- Grid property distributions (porosity, permeability, pressure, saturation)
- Spatial structure (cross-sections, maps)
- Train vs validation vs test distribution checks
- Any anomalies or patterns that matter for modeling

Write the code so it loads from parquet each time (no dependency on notebook 01 variables).
Use `display()` for matplotlib figures. Make plots publication-quality with proper labels.
""")

In [0]:
answer = ask_brain("""
I've completed notebook 02 EDA (you can see the full code in the project files). All 15 code cells ran successfully. Here are the KEY NUMERICAL RESULTS from the exported tables:

## PARAMETER STATS (100 cases)
- Fault Transmissibility: mean=0.100, std=0.030, range=[0.050, 0.150]
- Porosity Multiplier: mean=1.148, std=0.204, range=[0.801, 1.494]
- Permeability Multiplier: mean=5.200, std=2.727, range=[0.521, 9.785]
- Aquifer Pore Volume: mean=125.2, std=42.4, range=[54.0, 199.8]

## KS TESTS (split shift)
All p-values > 0.5 (lowest: Aquifer PV test split p=0.526). Max mean shift = 0.30 SD. Only 0-2 held-out cases outside training range per parameter.

## SPEARMAN CORRELATIONS (param → cumulative production, ALL 100 cases)
- Porosity Multiplier → oil_cum: 0.993, gas_cum: 0.992, water_cum: -0.995
- Fault Transmissibility → oil: 0.14, gas: 0.14, water: -0.14
- Permeability Multiplier → oil: 0.09, gas: 0.09, water: -0.01
- Aquifer Pore Volume → oil: -0.08, gas: -0.08, water: 0.14

## GRID-INPUT REDUNDANCY (training Spearman: param → grid summary)
- Porosity Multiplier → PORO_mean: 1.000 (PERFECT), oil_reservoir_volume_proxy: 1.000
- Permeability Multiplier → PERMX_geometric_mean: 1.000 (PERFECT)
- Aquifer Pore Volume → PORV_total: 0.836, PRESSURE_pv_weighted: 1.000, SWAT_pv_weighted: 1.000
- Fault Transmissibility → all grid summaries < 0.21 (NOT captured by grid statistics)

## PCA (training curves)
- oil_cum: 1 component = 99.95% variance
- gas_cum: 1 component = 99.96% variance  
- water_cum: 1 component = 99.91% variance

## HISTORY MATCH (top 15 cases)
- Best: Case 69 (NRMSE=0.164), Case 24 (0.164), Case 47 (0.164)
- All top cases have Porosity Multiplier ~1.2-1.3, wide range of Perm Mult and Aquifer PV
- NRMSE range across all 100: ~0.164 to ~0.45
- Oil/gas rate NRMSE ~0.21 for best cases; water rate NRMSE ~0.07 (better fit)

## CUMULATIVE PRODUCTION AT 2008
- Oil: mean=35.9M STB, std=2.9M, range=[29.9M, 40.2M]
- Gas: mean=13.0M MSCF, std=1.0M, range=[10.9M, 14.6M]
- Water: mean=14.7M STB, std=2.3M, range=[11.0M, 19.1M]

## GRID QUALITY
- 42,512 active cells per case (consistent), 0 duplicate cells
- PORO cell-ratio std across cases ~5e-6 (near-perfect uniform scaling)
- PERMX cell-ratio std ~0.42 (heterogeneous scaling!)

## PRODUCTION ANOMALIES
- 1.1M invalid value rows per metric (NaN in long format — expected)
- 0 negative values, 0 cumulative decreases (clean data)

Please provide a COMPLETE INTERPRETATION covering:
1. What the parameter space design tells us
2. Split quality assessment
3. Production sensitivity analysis (why Porosity Multiplier dominates)
4. History match implications
5. The grid redundancy finding and what it means architecturally
6. The PCA finding and what it means for model complexity
7. Production physics (GOR, water cut behavior)
8. **Specific modeling recommendations** — architecture, inputs, outputs, training strategy

Cite the actual numbers. This guides our model architecture decisions.
""")

In [0]:
answer = ask_brain("""
We're building a STACKED ENSEMBLE for MWAP DataWorks 2026. Team of 4 people — each develops one model family, then we stack them.

Here are the completed EDA findings (notebook 02 ran successfully):

KEY NUMBERS:
- 100 cases (train 1-70, val 71-85, test 86-100), 4 inputs → cumulative oil/gas/water curves
- Porosity Mult dominates: Spearman 0.993 oil, -0.995 water. Other 3 params <0.21
- PCA: 1 component = 99.95% cumulative variance (extremely low-rank)
- Grid summaries redundant with 4 inputs EXCEPT Fault Trans (<0.21 with all grid stats)
- PORO uniform scaling (std 5e-6), PERMX heterogeneous (std 0.42)
- Best history match Case 69 (NRMSE 0.164), top cases at Porosity ~1.2-1.3
- Oil 35.9M±2.9M STB, Gas 13.0M±1.0M MSCF, Water 14.7M±2.3M STB at 2008
- Water cut 61% at history end. Undersaturated reservoir, no free gas
- Forecast needed: quarterly cumulative to 2028 for 10 selected cases

Design a notebook '02b Model Selection EDA' that:

1. TESTS MODEL HYPOTHESES with quick baseline fits on ACTUAL data:
   - Fit 6-8 candidate models on train (1-70), score on val (71-85)
   - Use PCA coefficients as targets (since 1 component = 99.95%)
   - Compare: Ridge, Polynomial Ridge, Gaussian Process (ARD), XGBoost, Random Forest, MLP, KNN, SVR
   - Report val RMSE for each in physical units (STB, MSCF)
   - Also test: does adding grid features beyond 4 params help?

2. TESTS NONLINEARITY AND INTERACTIONS:
   - Residual analysis after linear fit — are residuals structured?
   - Partial dependence plots for each input
   - Interaction detection (does Fault Trans × Perm Mult matter?)

3. TESTS TEMPORAL/FORECASTING STRUCTURE:
   - Can a model trained on history extrapolate to 2028?
   - Rolling-origin backtest: train on 1998-2003, predict 2003-2008
   - Rate decline analysis: Arps decline curve fits
   - Does the temporal forecasting need a separate model from the historical surrogate?

4. RECOMMENDS EXACTLY 4 MODEL FAMILIES with justification:
   - Each must bring DIFFERENT inductive bias to the ensemble
   - Consider: one smooth interpolator, one tree-based, one physics-informed, one flexible nonlinear
   - Assign complexity levels (simple→complex) across the 4 team members

5. DESIGNS THE STACKING ARCHITECTURE:
   - How to combine 4 base models
   - Meta-learner choice
   - Cross-validation scheme for 70 training cases
   - How to handle the temporal forecasting component

For each cell: descriptive title, COMPLETE runnable Python code (reads from /Workspace/Users/tengkuanas.zainalabi@petronas.com.my/MWAP/data/), what insight it reveals.
Save figures to /Workspace/Users/tengkuanas.zainalabi@petronas.com.my/MWAP/outputs/model_selection/figures/.
Use sklearn, xgboost, matplotlib/seaborn. Publication-quality plots.
""", max_tokens=65000)

In [0]:
answer = ask_brain("""
Both EDA notebooks have now run successfully. Interpret the COMBINED results from notebook 02 (general EDA) and notebook 02b (model selection EDA).

From 02b, the key results are:

## BASELINE HORSE RACE (8 models, val cases 71-85)
Ranked by mean_phase_NRMSE:
1. Polynomial Ridge: 0.004008
2. Gaussian Process ARD: 0.004084
3. MLP: 0.009764
4. XGBoost: 0.010509
5. Ridge: 0.012456
6. Random Forest: 0.012882
7. SVR: 0.021992
8. KNN: 0.030800

## PCA RANK
- All models used 3 PCs per phase (0.9999 threshold)
- GP with PCA=1 (1 PC): water endpoint RMSE 42K STB
- GP with PCA=0.9999 (3 PCs): water endpoint RMSE 96K STB (worse — overfitting higher PCs!)

## GRID ABLATION
- 4 params only vs 4 params + grid: compared with Polynomial Ridge and XGBoost
- Grid features add what? Report the numbers you see.

## OOF CROSS-VALIDATION (5-fold, cases 1-70)
Pooled OOF NRMSE:
1. Polynomial Ridge: best
2. Gaussian Process ARD: close second
3. Ridge: ~0.012
4. MLP and XGBoost: ~0.03 (much worse in CV than validation!)

## LINEAR RESIDUAL DIAGNOSTICS
- All p-values > 0.27 — NO significant nonlinear residual structure detected
- Fault Transmissibility squared, Fault x Permeability: all p > 0.8

## INTERACTION ABLATION
- Quadratic main effects CV NRMSE: ~0.005
- Main effects + Fault x Perm: ~0.005 (no improvement)
- All quadratic interactions: ~0.004 (best)

## TEMPORAL BACKTESTS (rolling origin: 2003, 2004, 2005)
- Constant rate, Exponential, Arps all tested
- Increment NRMSE at 5yr horizon: ~0.39-0.41 (poor!)
- At 3yr horizon: ~0.20 (moderate)
- Exponential and Arps essentially identical (b parameter goes to ~0)

## PILOT STACK
Simplex weights: GP=0.290, Poly Ridge=0.292, XGBoost=0.207, MLP=0.211
Stack validation vs individual: stack improves over uniform average

## 2028 ASSUMPTION SENSITIVITY
- Constant rate vs Exponential give hugely different 2028 endpoints
- Oil: 85.9M (constant) vs 57.4M (exponential) — a 50% difference!

Please provide a COMPLETE INTERPRETATION combining 02 and 02b findings:

1. What the model selection EDA conclusively proved
2. Whether the 4 recommended families (Physics-informed, GP, XGBoost, MLP) are correct given the data
3. Key surprises (e.g., why did GP with more PCs perform WORSE?)
4. The temporal forecasting problem — is it fundamentally harder than history matching?
5. Specific recommendations for each team member's implementation
6. Stacking strategy refinements based on the pilot results
7. Risks and open questions heading into notebook 03
""", max_tokens=65000)

In [0]:
answer = ask_brain("""
Provide a CELL-BY-CELL interpretation of EVERY analysis from BOTH notebooks 02 and 02b.

## Notebook 02 EDA — cover each of these analyses with ACTUAL NUMBERS:
1. Parameter distributions — ranges, means, stds for all 4 uncertainty params
2. Parameter correlations — inter-parameter independence
3. Parameter space — coverage, boundary cases
4. Split shift — KS p-values, mean shifts train vs val vs test
5. Production curve families — spread of oil/gas/water across 100 cases
6. Parameter-production Spearman correlations — the 0.993 porosity dominance
7. Parameter-endpoint scatter
8. History match ranking — Case 69 NRMSE 0.164, what the best cases share
9. Best case rate overlays
10. GOR/water cut consistency — solution GOR 0.3633
11. Grid profiling — 127 features from 42,512 cells, quality checks
12. Grid distributions by split
13-16. Reservoir maps, cross-sections, footprint, layer profiles
17. Grid-input redundancy — perfect correlations except Fault Trans
18. Curve dimensionality PCA — 99.95% in 1 PC
19. Modeling handoff summary

## Notebook 02b Model Selection EDA — cover each cell:
1. Data audit — 943,160 rows per metric, 14,645 finite values
2. CurveSurrogate design — PCA + StandardScaler + model, 3 PCs per phase
3. Horse race — PolyRidge 0.004, GP 0.004, MLP 0.010, XGB 0.011, Ridge 0.012, RF 0.013, SVR 0.022, KNN 0.031
4. PCA floor — 1 PC reconstruction vs 3 PCs
5. GP rank ablation — 1 PC water RMSE 42K vs 3 PCs 96K (WORSE)
6. Grid ablation — 4 params vs 4+grid for PolyRidge and XGBoost
7. OOF CV — 5-fold scores, PolyRidge and GP lead, MLP/XGB much worse
8. Residual diagnostics — all p > 0.27, no significant nonlinearity
9. Partial dependence — GP response curves per input
10. Interaction ablation — quadratic main 0.005, +Fault*Perm 0.005, full quad 0.004
11. Temporal models — Arps decline, water plateau, GOR coupling
12. Rolling backtests — 5yr increment NRMSE ~0.40, 3yr ~0.20
13. 2028 sensitivity — oil 85.9M constant vs 57.4M exponential (50% gap!)
14. Pilot stack — weights GP=0.29, PolyRidge=0.29, XGB=0.21, MLP=0.21
15. Final 4 families + stacking contract

For EACH cell give the actual key numbers and what they mean.

End with:
1. SO WHAT — unified conclusion about the reservoir and what ML can achieve
2. Biggest risk heading into notebook 03
3. One-paragraph executive summary for a non-technical manager
""", max_tokens=65000)

In [0]:
answer = ask_brain("""
Two questions. Be direct and decisive.

QUESTION 1: IS THE EDA SUFFICIENT?

We've completed notebook 02 (general EDA) and 02b (model selection EDA). Here's everything we now know:

- 100 cases, 4 inputs, 3 cumulative targets (oil/gas/water)
- Porosity dominates (Spearman 0.993), other 3 params <0.21
- PCA: 1 PC = 99.95% variance — extremely low-rank
- Grid features redundant with 4 inputs except Fault Trans
- Horse race: PolyRidge (0.004) ≈ GP (0.004) >> MLP (0.010) > XGB (0.011)
- OOF CV confirms PolyRidge and GP lead; MLP/XGB unstable (0.03 in CV vs 0.01 on val)
- No significant nonlinear residual structure (all p > 0.27)
- Full quadratic interactions help (0.004 vs 0.005 main effects only)
- Fault×Perm interaction specifically: NO improvement
- Grid ablation: tested 4 params vs 4+grid
- Temporal backtests: 5yr increment NRMSE ~0.40 (100x worse than historical surrogacy)
- 2028 oil: 85.9M (constant) vs 57.4M (exponential) — 50% divergence
- Pilot stack weights near-uniform (likely shrinkage-driven)
- GP with 1 PC beats GP with 3 PCs (overfitting higher modes)

Do we have ENOUGH information to confidently assign 4 model families to 4 team members and start building? Or is there a critical gap that would change our architecture decisions?

Be honest — if more EDA is needed, say what specifically. If we're ready, say so.

QUESTION 2: WHAT SHOULD THE 4 FAMILIES BE?

Given ALL the evidence from 02 and 02b, what are your final recommended 4 model families?

For each family:
1. Name and one-sentence description
2. WHY this family given the specific data characteristics we found
3. What it brings to the ensemble that the others don't
4. Key implementation decisions (PCA rank, regularization, architecture)
5. Expected difficulty level (easy/medium/hard) and estimated dev time
6. The single biggest risk for this family

Also address:
- Should we keep PolyRidge as a standalone family or fold it into the physics model?
- Is XGBoost worth a team member given its poor OOF stability?
- How should we handle the temporal forecasting problem — does ONE family own it, or does everyone build their own temporal decoder?
- Assignment suggestion: which family goes to which complexity level (for team members with different ML experience)?
""", max_tokens=65000)

In [0]:
answer = ask_brain("""
An external AI reviewed our 4-family plan and gave a sharp critique. I need you to STRESS TEST this critique objectively. Fight with it where it's wrong, concede where it's right. Be brutally honest.

Here is the external AI's argument:

---
CRITIQUE:

1. "Your plan breaks the end-to-end rule. Family 3 owns the entire 20-year forecasting problem while Families 1, 2, 4 are pure history-match emulators. If Family 3 fails at temporal forecasting, your entire team fails because nobody else builds a forecast. Everyone needs to do both."

2. "It turned your team into a model-fitting contest. MLP (Family 4) degrades 3x in cross-validation and overfits with 70 cases. Bringing back an MLP just to 'earn its ensemble weight' is a waste of days."

3. "It missed the core bottleneck. Your EDA said the gap between emulating the simulator and predicting the future is 100x. Stopping at PolyRidge and GP only solves the easy 0.4% error. It leaves the 20-year future completely unaddressed."

4. "Everyone should build an end-to-end pipeline (history match + 20-year forecast), but use a DIFFERENT mechanism to bridge past to future:"
   - Family 1: PolyRidge history + Classical Arps/Exponential Decline anchored at 2008
   - Family 2: GP history + Stochastic P10/P50/P90 future curves from GP variance
   - Family 3: Ridge + explicit Fault Trans feature extraction + Bounded Hyperbolic Decline tuned to fault mechanics
   - Family 4: Baseline emulator + VECM/SINDy/Hard-coded constraints (water cut <96%)
---

Now STRESS TEST this critique against our ACTUAL EDA evidence. For each of the 4 arguments:

1. Is the critique FACTUALLY CORRECT about what our plan says?
2. Is the critique's REASONING sound given our specific data?
3. Where is the critique WRONG or misleading?
4. Where is the critique RIGHT and we should listen?

Specifically address:
- Does the critique's alternative family table actually make sense for OUR data (100 cases, 4 inputs, porosity-dominated, 1 PC = 99.95%)?
- Is "Ridge + Fault Trans feature extraction + Bounded Hyperbolic Decline tuned to fault mechanics" even meaningful when Fault Trans has Spearman <0.21 with everything?
- Is VECM/SINDy appropriate for 70 training cases with 4 inputs?
- Does "GP variance as P10/P50/P90 forecast bounds" actually work when GP uncertainty is NOT calibrated forecast uncertainty?
- Is the MLP critique fair? Our OOF showed 0.03 vs 0.01 on validation.
- Is the "everyone forecasts" approach better than "shared temporal decoder"?

End with YOUR FINAL VERDICT:
- What should we CHANGE about our plan based on this critique?
- What should we REJECT?
- What's the updated 4-family recommendation?

Be objective. If the external AI is right, say so. If it's wrong, explain why with our actual numbers.
""", max_tokens=65000)

In [0]:
answer = ask_brain("""
We're building Anas's workstream (Workstream 4: Alternative Temporal Structure & Forecast Validation).

The team folder is: /Workspace/Users/tengkuanas.zainalabi@petronas.com.my/mwap-exdw-surrogate-model/Anas-db/
Shared data lives in: /Workspace/Users/tengkuanas.zainalabi@petronas.com.my/MWAP/data/ (uncertainty_params.parquet, production_timeseries.parquet, grid_properties.parquet)
Shared outputs from notebooks 02 and 02b are in: /Workspace/Users/tengkuanas.zainalabi@petronas.com.my/MWAP/outputs/

Anas's role is:
- Owner of RIGOROUS FORECAST VALIDATION and alternative temporal structures
- Challenges the default temporal decoder (exponential decline + water plateau)
- Tests alternative formulations: total-liquid rate + water-cut dynamics vs separate oil/water declines
- Runs crossed case×time backtests to determine which forecast assumptions survive
- Ensures the team doesn't submit a 40%-error forecast
- Also produces his own end-to-end pipeline (history match + forecast) using PolyRidge or GP as regressor

REMINDER OF KEY EDA FACTS:
- 100 cases, train 1-70, val 71-85, test 86-100
- 4 inputs → 3 cumulative targets (oil/gas/water), quarterly, history 1998-2008
- Porosity dominates (0.993), other 3 params < 0.21
- PolyRidge and GP both at 0.004 NRMSE (history)
- Temporal backtests: 5yr increment NRMSE ~0.40, 3yr ~0.20
- 2028 oil: 85.9M (constant) vs 57.4M (exponential) — 50% divergence
- Arps b→0, so exponential ≈ Arps
- GP with 1 PC beats 3 PCs
- GOR constant at 0.363 MSCF/STB (undersaturated)
- Water cut ~61% at 2008

Design TWO notebooks for the Anas-db/ folder:

## NOTEBOOK 1: `Anas-db/01 Data Preparation.ipynb`
What Anas-specific data prep is needed beyond the shared parquets?
- Load shared parquets and create Anas's working datasets
- Build the cutoff-safe temporal targets (rate at cutoff, cumulative anchor, fitted decline parameters)
- Create the crossed case×time evaluation framework (cases × cutoff dates)
- Extract temporal features: late-history rates, rate derivatives, water-cut trajectory
- Build the "total liquid + water cut" alternative representation
- Save everything to Anas-db/data/ as parquets

For each cell: title, COMPLETE runnable Python code, what it produces.

## NOTEBOOK 2: `Anas-db/02 Temporal EDA.ipynb`
This is the FAMILY-SPECIFIC EDA — not repeating notebook 02, but diving deep into temporal structure:
- How do rate decline parameters vary across cases? Are they identifiable?
- fit_temporal() parameter sensitivity: how stable are D, b, water_plateau across fitting windows?
- Total-liquid-rate vs separate oil/water: which representation is more stable?
- Water-cut dynamics: is water cut better modeled as a function of cumulative liquid?
- Rolling-origin backtest deep dive: WHERE do the errors concentrate (early vs late forecast)?
- Can the 4 uncertainty params predict temporal parameters? (Spearman, scatter plots)
- Parameter identifiability: do very different decline params produce similar histories?
- Comparison: constant-rate vs exponential vs total-liquid+WC for 3yr and 5yr horizons
- Figures saved to Anas-db/outputs/figures/, tables to Anas-db/outputs/tables/

For each cell: title, COMPLETE runnable Python code, what insight it reveals.

All code must:
- Read from the shared /MWAP/data/ parquets
- Be fully self-contained and runnable
- Use matplotlib/seaborn for publication-quality plots
- Use display() for showing figures
- Save artifacts to Anas-db/data/ or Anas-db/outputs/
""", max_tokens=65000)

In [0]:
answer = ask_brain("""
Anas's 02 Temporal EDA notebook just ran successfully (all 13 cells). The outputs are in:
- Anas-db/outputs/figures/ (10+ publication-quality PNG+PDF figures)
- Anas-db/outputs/tables/ (15+ CSV tables)
- Anas-db/data/ (parquets: crossed_backtest_predictions, crossed_parameter_prediction_errors, conditional_2028_scenarios, anas_historical_predictions.npz)
- Anas-db/outputs/TEMPORAL_VALIDATION_HANDOFF.md

Here are the KEY NUMERICAL RESULTS from the executed cells:

## TEMPORAL PARAMETER DISTRIBUTIONS (Cell 3)
- 94% of water plateau fits hit upper bound at 2003 cutoff (only 3yr water history)
- D_oil and D_liquid distributions vary significantly across cutoffs
- All cutoff observation ages = 0 days (no stale data issue)

## FITTING-WINDOW SENSITIVITY (Cell 4)
- Parameters change materially between 3yr, 5yr, and all-history windows
- Exported: 02_window_sensitivity_summary.csv, 02_window_sensitivity_by_case.csv

## WATER CUT COORDINATE COMPARISON (Cell 5)
- Cumulative liquid coordinate: median WC RMSE = 0.0085, p90 = 0.0133
- Time coordinate: median WC RMSE = 0.0090, p90 = 0.0136
- Cumulative liquid is SLIGHTLY better but the difference is small

## INPUT-TEMPORAL PARAMETER CORRELATIONS (Cell 6)
- Spearman heatmaps at 2003 and 2008 cutoffs
- Scatter plots for D_oil, D_liquid, water_ratio, beta_volume vs all 4 inputs

## PARAMETER IDENTIFIABILITY (Cell 7)
- Jacobian condition numbers: median ~6e15 (EXTREMELY ill-conditioned!)
- Near-equivalent history fits with very different future projections
- Arps b diagnostic confirms b -> 0

## CROSSED CASE×TIME BACKTESTS (Cells 8-9)
- 114,240 prediction rows across 4 experiments, 4 methods, 2 modes (history-conditioned + input-only)
- Leaderboard: macro scores sorted by mean_phase_increment_NRMSE
- Paired decoder comparisons: Constant vs Separate vs Liquid-WC-time vs Liquid-WC-volume
- Parameter map prediction scores by target

## ERROR CONCENTRATION (Cell 10)
- Early vs late half error breakdown
- Error by forecast lead quarter
- Endpoint errors vs porosity for each method

## ANAS HISTORICAL SURROGATE (Cell 11)
- oil_cum training NRMSE: 0.00245 (better than shared 02b's 0.004!)
- gas_cum: similar
- Negative rate fraction: 0.77% (train), 1.19% (validation) — minor physical violations
- Diagnostic history-match ranking produced

## 2028 SCENARIOS (Cell 12)
- 4 methods applied to validation cases 71-85
- Structural spread between methods documented
- 09_2028_structural_spread.csv shows the divergence

## FORECAST REVIEW FLAGS (Cell 13)
- Warning flag: review_required_over_020 = True for most experiments
- Development decoder ranking produced
- TEMPORAL_VALIDATION_HANDOFF.md exported

Please provide a COMPLETE INTERPRETATION of the Temporal EDA:

1. For each analysis cell, what does the result MEAN for our forecasting strategy?
2. Which temporal decoder won the crossed backtests? (Constant vs Separate vs Liquid-WC-time vs Liquid-WC-volume)
3. Is the water-cut-in-cumulative-liquid approach actually better?
4. How bad is the identifiability problem? What should we do about it?
5. Does the historical surrogate (0.00245 NRMSE) beat the shared 02b baseline (0.004)?
6. What's the final recommendation for Anas's workstream heading into notebook 03?
7. What are the 3 most important findings that should change how the WHOLE team builds their models?

Be specific with numbers from the actual outputs.
""", max_tokens=65000)

In [0]:
answer = ask_brain("""
An external AI has proposed a plan for Anas's next notebook: "03 Family-Specific EDA" for Workstream 4 (Alternative Temporal Structure & Validation). The external AI argues the previous EDA was general-purpose and now Anas needs an EDA 100% focused on time, decay physics, and stress-testing the future.

Here is the external AI's proposed plan:

---
PROPOSED ANALYSIS 1: Cutoff Sensitivity Test (Rolling Backtests)
- Truncate historical data at 2003, 2005, and 2006 for a few cases
- Fit the baseline temporal decoder to each cutoff, project forward to 2008
- If 2003 projection wildly misses actual 2008 data, proves temporal momentum is unstable
- Gives baseline error rate for how fast predictions decay

PROPOSED ANALYSIS 2: b-Value Ablation
- Plot future production curve using unconstrained Arps equation
- Then plot again using strict Exponential decline (b -> 0)
- Visually demonstrates the 6e15 condition number problem
- Shows unconstrained line doing something physically unrealistic

PROPOSED ANALYSIS 3: Error Compounding Trajectory
- Calculate forecast increment NRMSE month-by-month over a 5-year gap (not single aggregated number)
- Shows exactly WHEN models break: immediate spike after cutoff or slow drift by year 3?
---

Now here is what notebook 02 Temporal EDA ALREADY produced (all ran successfully):
- 4 cutoff origins tested: 2003, 2004, 2005, 2008
- 4 temporal decoder methods compared: Constant, Separate, Liquid-WC-time, Liquid-WC-volume
- 114,240 crossed backtest prediction rows across 4 experiments x 4 methods x 2 modes
- Multistart identifiability diagnostics with 1,680 fits confirming b->0 and Jacobian ~6e15
- Window sensitivity (3yr, 5yr, all) already tested
- Water cut coordinate comparison (time vs cumulative liquid)
- Input-temporal parameter correlations
- Historical surrogate NRMSE 0.00245
- Error by forecast lead quarter already computed
- Early vs late half error breakdown already computed
- 2028 conditional scenarios already generated
- Paired decoder comparison tables exported
- Development decoder ranking exported

I need you to OBJECTIVELY stress test the external AI's plan:

1. For EACH of the 3 proposed analyses: Is this ALREADY DONE in notebook 02? If partially done, what's the delta? If not done, is it worth doing?

2. Is the external AI right that the previous EDA was too general? Or did notebook 02 actually already focus heavily on Anas's W4 domain (temporal structure, decoder comparison, forecast validation)?

3. What should notebook 03 ACTUALLY be? Should it be:
   a) More family-specific EDA (as external AI suggests)?
   b) The actual modeling notebook (forecast-first design, as Brain previously recommended)?
   c) Something else entirely?

4. If notebook 03 IS needed as a family-specific EDA, propose a CONCRETE cell-by-cell plan that:
   - Does NOT duplicate what 02 already did
   - Adds genuine new insight for decoder selection
   - Is completable in <15 cells
   - Uses the actual data artifacts already in Anas-db/data/ and Anas-db/outputs/tables/

5. If notebook 03 should skip EDA and go straight to modeling, explain WHY and what the modeling notebook should contain.

Be brutally honest. If the external AI is proposing redundant work, say so. If the external AI has a point we missed, concede it. The goal is to make the BEST use of Anas's limited time before the competition deadline.
""", max_tokens=65000)

In [0]:
answer = ask_brain("""
Design notebook `Anas-db/03 Forecast Modeling & Validation.ipynb` — the ACTUAL MODELING notebook for Workstream 4.

Here are the ACTUAL NUMERICAL RESULTS from the completed 02 Temporal EDA:

## DECODER RANKING (development winner)
| Method | Mean Dev Increment NRMSE | Worst Experiment |
|---|---|---|
| Separate | 0.1548 | 0.2935 |
| Liquid-WC-volume | 0.2117 | 0.4448 |
| Liquid-WC-time | 0.2210 | 0.4392 |
| Constant | 0.2756 | 0.4037 |

**Separate WINS by a large margin.** Liquid-WC methods are 37-43% worse. Constant is worst overall.

## KEY EXPERIMENT SCORES (Input-only PolyRidge, Separate method)
- 2003→3yr: increment NRMSE 0.205 (train OOF), 0.203 (validation)
- 2003→5yr: increment NRMSE 0.293 (train OOF), 0.294 (validation)
- 2004→3yr: increment NRMSE 0.064 (train OOF), 0.070 (validation)
- 2005→3yr: increment NRMSE 0.056 (train OOF), 0.060 (validation)

**Pattern: Later cutoffs have MUCH lower NRMSE.** 2004/2005 are 3-5x better than 2003.

## HISTORICAL SURROGATE (Anas, PolyRidge, 2008 cutoff)
| Metric | Train NRMSE | Val NRMSE |
|---|---|---|
| oil_cum | 0.00245 | 0.00384 |
| gas_cum | 0.00243 | 0.00377 |
| water_cum | 0.00806 | 0.00642 |
| oil_rate | 0.00980 | 0.00945 |
| gas_rate | 0.00969 | 0.00935 |
| water_rate | 0.02151 | 0.01905 |

## PHYSICAL VIOLATIONS
| Eval | Neg Cum | Decreasing Steps | Neg Rates |
|---|---|---|---|
| Train OOF | 0.91% | 0.87% | 0.77% |
| Validation | 1.41% | 1.28% | 1.19% |

## 2028 STRUCTURAL SPREAD (validation cases)
- Oil: 55-57% spread relative to minimum across methods
- Gas: 55-57% spread
- Water: 44-50% spread

## PARAMETER MAP PREDICTION (key targets, transformed RMSE)
- Anchors (oil/gas): 0.002 (excellent)
- Anchor water: 0.007-0.009 (good)
- D_oil (Separate): 0.076 at 2003, improves at later cutoffs
- water_k: 0.119 at 2003 (hardest temporal target)
- water_ratio: 0.099 at 2003
- D_liquid (Liquid methods): 1.613 at 2003 (TERRIBLE — explains why Liquid methods lose)

## CRITICAL INSIGHT
D_liquid prediction RMSE = 1.613 at 2003 explains EVERYTHING about why Liquid-WC methods fail. The liquid decline rate is unpredictable from inputs. Separate method's D_oil (0.076) is much more learnable.

Now design the COMPLETE notebook 03. Requirements:

1. This is the END-TO-END forecasting notebook — history match + 2028 forecast
2. Uses Separate decoder as the baseline (it won)
3. Uses PolyRidge as the primary regressor (proven at 0.004 NRMSE)
4. Tests GP as a controlled alternative
5. Addresses the 3 known issues: (a) physical violations ~1%, (b) water targets harder to predict, (c) 2003 forecast still at 0.20 NRMSE
6. Produces DEPLOYABLE predictions for Cases 71-85 (validation) at the 2008 cutoff
7. Also generates conditional 2028 forecasts
8. Exports everything the team needs for integration

For EACH cell, provide:
1. Cell title
2. COMPLETE, RUNNABLE Python code (no pseudocode)
3. Expected outputs

The notebook must:
- Load from Anas-db/data/ parquets (produced by notebook 01)
- Import temporal_utils from Anas-db/temporal_utils.py
- Use the same case splits: train 1-70, val 71-85, test 86-100
- Use 5-fold CV on train (folds from case_folds.parquet)
- Save outputs to Anas-db/outputs/ and Anas-db/data/
- Be <15 cells total (title + pip + 12-13 code cells)

IMPORTANT: The code must be self-contained and runnable. Use the EXACT same paths and conventions as notebooks 01 and 02. Reference temporal_utils functions (target_from_history, encode_targets, decode_targets, forecast, etc.).
""", max_tokens=65000)

In [0]:
answer = ask_brain("""
Anas's notebook 03 Forecast Modeling & Validation has been BUILT AND RUN SUCCESSFULLY — all 14 cells passed. This is the END-TO-END forecasting notebook for Workstream 4.

Here is the COMPLETE structure and ALL KEY NUMERICAL RESULTS:

## NOTEBOOK STRUCTURE (14 cells, all passed)
1. Title markdown
2. pip install
3. Config + imports + schema validation (temporal_utils hash: 3ba3b98c)
4. Build input matrix + encode Separate targets (X_train: 70×4, z_train: 70×9)
5. Define PolyRidge + GP candidates (StandardScaler → PolynomialFeatures(2) → Ridge(α=1) vs StandardScaler → Matern(ν=2.5) ARD GP)
6. 5-fold OOF predictions at 2008 cutoff
7. Crossed forecast backtests (4 experiments × 2 models × train OOF + validation)
8. Forecast scoring and model comparison
9. Physical validity corrections
10. Lead-time error analysis
11. Fit locked model on full training set
12. Historical surrogate and history-match ranking
13. Deployable 2028 conditional forecasts (validation cases 71-85)
14. Model serialization and team handoff

## CRITICAL RESULT: MODEL RANKING
| Model | mean_dev_NRMSE | worst_exp_NRMSE |
|---|---|---|
| GP | 0.152122 | 0.301434 |
| PolyRidge | 0.152544 | 0.293586 |

**GP WINS by 0.0004 on mean development NRMSE.** However PolyRidge has a BETTER worst-case (0.294 vs 0.301).
The locked model is GP.

## MACRO SCORES (all experiments, both models)
Experiments tested: 2003→3yr, 2003→5yr, 2004→3yr, 2005→3yr
Both PolyRidge and GP tested across all 4 experiments with train_OOF and validation splits.
GP and PolyRidge are EXTREMELY close across all experiments.

## PHYSICAL VALIDITY
- GP: negative_prediction_fraction = 0.0, corrected_fraction = 0.0 for BOTH train_OOF and validation
- PolyRidge: similarly 0.0
- ZERO physical violations in the forecast-first pipeline (unlike notebook 02's ~1% violations)

## TARGET PREDICTION QUALITY (GP, 2008 cutoff, full training fit)
Head-by-head residuals (validation):
- D_oil: mean_abs_error ~4e-4 (excellent)
- anchor_gas: mean_abs_error ~2.39 (in original scale)
- anchor_oil, anchor_water: very small errors
- water_ratio, water_k: small errors

## HISTORICAL SURROGATE
- Short-horizon check (2008+1Q): "No historical scores computed (forecast errors)" — the forecast() function raised exceptions for some cases at the 1-quarter horizon. This suggests the decoded temporal parameters have edge cases where the integrator fails at very short leads.

## 2028 ENDPOINT FORECASTS (validation cases 71-85)
Descriptive stats across 15 cases:
- oil_cum: mean ~4-6×10^7 (with spread across cases)
- gas_cum: mean ~1.2-2.0×10^7
- water_cum: significant spread, some cases showing ~1.2×10^8
All 15 validation cases produced physically plausible monotonic cumulative curves.

## FORECAST FAN CHARTS (from the figure)
The 2028 forecast fans show:
- oil_cum: Well-separated cases, smooth concave curves, no crossings — clean uncertainty resolution
- gas_cum: Similar pattern, well-behaved
- water_cum: MOST spread between cases, some cases with steep late-time acceleration (high-water scenarios)

## EXPORTED ARTIFACTS
- data/forecast_modeling_backtests.parquet (28,560 rows)
- data/forecast_modeling_2028.parquet (3,600 rows — 15 val cases × 80 dates × 3 phases)
- outputs/forecast_modeling/models/ (9 head_*.joblib + 2 scalers)
- outputs/forecast_modeling/locked_specification.json
- outputs/forecast_modeling/tables/ (phase_scores, macro_scores, model_ranking, physical_validity, lead_time_scores, early_vs_late_scores, target_prediction_quality, historical_surrogate_scores, history_match_ranking, validation_2028_endpoints)
- outputs/forecast_modeling/figures/ (forecast_error_by_lead.png/pdf, validation_2028_fans.png/pdf)
- outputs/forecast_modeling/FORECAST_MODELING_HANDOFF.md

## CONTEXT FROM NOTEBOOK 02 (for comparison)
- Notebook 02 Temporal EDA found Separate decoder wins at 0.155 mean dev NRMSE
- Notebook 02 had ~1% physical violations (neg cum, decreasing steps, neg rates)
- Notebook 02 historical surrogate: oil_cum NRMSE 0.00245 (train), 0.00384 (val)

Please provide a COMPLETE INTERPRETATION:

1. **GP vs PolyRidge verdict**: GP won by 0.0004 on mean but PolyRidge has better worst-case. Is this meaningful? Should we stick with GP or reconsider?

2. **Physical validity improvement**: Notebook 02 had ~1% violations, notebook 03 has ZERO. What changed and why?

3. **Historical surrogate failure**: The short-horizon check failed with "No historical scores computed (forecast errors)". What does this mean? Is this a bug in the forecast() function at 1-quarter leads, or a real modeling issue?

4. **2028 forecast fans**: Interpret the fan charts. The water_cum has the most spread — is this expected? Are these physically plausible curves?

5. **28,560 backtest rows vs 114,240 in notebook 02**: Notebook 02 tested 4 methods × 2 modes (history-conditioned + input-only), notebook 03 tests 2 models × input-only. Is this the right scope?

6. **Overall assessment**: Rate the notebook 03 output on a scale of 1-10 for:
   a) Scientific rigor (proper CV, no data leakage, test set untouched)
   b) Deliverable readiness (for the DataWorks competition)
   c) Team integration (can Nazrul/Afiq/Hazeem use these outputs?)

7. **What's missing or could be improved?** Any gaps before the team submission?

8. **Comparison to Brain's original 14-cell design**: How well did the actual notebook follow the Brain's design? What deviated and was the deviation justified?
""")

In [0]:
answer = ask_brain("""
STOP. Forget PolyRidge, forget GP, forget team integration. I am Anas, Workstream 4 owner.

My job is NOT to build the historical ML models. My job is:
- Timeline mechanic and referee
- Build the backtester and fix time-series bugs
- Build the strictest, most physically realistic future timeline

I need a FOCUSED EDA plan for MY specific family. This is NOT a general notebook — this is Anas exploring the time-series mechanics of decline curves before building anything.

Here is what I already have from notebook 02 Temporal EDA:
- 4 cutoff origins tested (2003, 2004, 2005, 2008)
- Separate decoder won (0.155 NRMSE) over Constant, Liquid-WC-time, Liquid-WC-volume
- Arps b → 0 confirmed across all cases
- Jacobian condition number ~6e15 (extremely ill-conditioned)
- Error by forecast lead quarter already computed (aggregate)
- 2028 structural spread: oil 55-57%, water 44-50%
- Water cases pushing ~120 million STB cumulative by 2028
- Physical violations: ~1% neg cum, ~1.3% decreasing steps, ~1.2% neg rates

Now design a CLEAN, FOCUSED notebook that ONLY explores these 3 analyses:

## ANALYSIS 1: Matched Ablation Test — Free-b Arps vs Strict b=0 Exponential
- For a representative subset of cases (say 10-15 spanning the porosity range)
- At the 2008 cutoff with 3yr fitting window:
  - Fit FREE Arps (unconstrained b) to the historical rates
  - Fit STRICT exponential (b=0 forced) to the same data
  - Project both forward 20 years to 2028
- Visualize: overlay the two forecast curves per case
- Quantify: how different are the 2028 endpoints? Does free-b produce physically unrealistic behavior?
- Show that the 6e15 condition number manifests as actual forecast divergence
- The key question: does constraining b=0 HELP or HURT forecast accuracy?

## ANALYSIS 2: Error Compounding Trajectory
- Using the crossed backtest predictions already in Anas-db/data/crossed_backtest_predictions.parquet (114,240 rows)
- Calculate forecast increment NRMSE QUARTER-BY-QUARTER (not aggregated)
- For each experiment (2003→3yr, 2003→5yr, 2004→3yr, 2005→3yr):
  - Plot NRMSE vs forecast lead time (quarters after cutoff)
  - Separate lines for oil_cum, gas_cum, water_cum
  - Show whether error is: (a) immediate spike after cutoff, (b) slow drift, or (c) late-time explosion
- This reveals exactly WHEN and WHERE the forecasts break
- Compare: does the error pattern differ between the Separate decoder and Constant decoder?

## ANALYSIS 3: Physical Water Bounds Diagnosis
- Using the 2028 conditional scenarios already in Anas-db/data/conditional_2028_scenarios.parquet
- For the high-water cases (top 5 by water_cum at 2028):
  - Plot water RATE over time (not just cumulative) — is the rate increasing, plateauing, or declining?
  - Calculate implied average water rate for the forecast period
  - Compare to 2008 observed water rate (~7,926 STB/day from the data)
  - Check: is 120M STB physically plausible given the aquifer pore volume parameter?
- Diagnose the water_k and water_ratio parameters for these cases
- Propose physical caps: what's a reasonable maximum water rate or cumulative?
- Show what happens if we apply log1p capping to the water dynamics

For EACH analysis, provide:
1. Cell title
2. COMPLETE RUNNABLE Python code
3. What figures/tables it produces
4. What insight it reveals for MY workstream

The notebook should be:
- ~8-10 cells total (1 title + 1 pip + 1 config/load + 5-7 analysis cells)
- Load from Anas-db/data/ parquets already produced by notebooks 01 and 02
- Import temporal_utils from Anas-db/temporal_utils.py
- Save figures to Anas-db/outputs/figures/ and tables to Anas-db/outputs/tables/
- Use matplotlib/seaborn for publication-quality plots
- NO PolyRidge, NO GP, NO sklearn — pure time-series mechanics
- Use the actual Arps decline equations directly

Remember: temporal_utils.py has these key functions:
- fit_trials(history, window, seed, n_starts) — multistart Arps+water fitting
- target_from_history(history, window) — extracts temporal params
- forecast(record, method, cutoff, dates) — integrates rates to cumulative
- encode_targets / decode_targets — log/logit transforms
- prefix_history(curves, case, cutoff) — restricts to date≤cutoff
- window_frame(history, window) — filters to 3y/5y/all
- METHODS, HEADS, PARAMS, CUM_METRICS, RATE_METRICS constants

This is MY focused exploration before I build anything. Help me understand the physics.
""", max_tokens=65000)

In [0]:
answer = ask_brain("""
Full status check for Anas, Workstream 4. Tell me what's next.

Here is EXACTLY where things stand:

## COMPLETED NOTEBOOKS (all cells passed)
1. `Anas-db/01 Data Preparation` — 9 cells, all passed. Produced:
   - uncertainty_params.parquet (85 cases × 6 cols)
   - case_folds.parquet (70 train, 5 folds × 14 each)
   - cutoff_history.parquet (9,860 rows)
   - temporal_targets.parquet (340 rows × 39 cols)
   - temporal_fits.parquet (1,020 rows)
   - forecast_truth.parquet
   - temporal_fit_trials.parquet
   - preparation_manifest.json
   - temporal_utils.py (hash 3ba3b98c)

2. `Anas-db/02 Temporal EDA` — 14 cells, all passed. Key findings:
   - Separate decoder wins (0.155 NRMSE) over Constant, Liquid-WC-time, Liquid-WC-volume
   - Arps b → 0 confirmed, Jacobian ~6e15
   - 94% water plateau fits hit upper bound at 2003
   - Historical surrogate: oil_cum NRMSE 0.00245 train, 0.00384 val
   - Physical violations: ~1% neg cum, ~1.3% decreasing, ~1.2% neg rates
   - 2028 structural spread: oil 55-57%, water 44-50%
   - Water pushing ~120M STB cumulative
   - Produced: crossed_backtest_predictions.parquet (114,240 rows), conditional_2028_scenarios.parquet, anas_historical_predictions.npz, 15+ CSV tables, 10+ figures, TEMPORAL_VALIDATION_HANDOFF.md

3. `Anas-db/03 Forecast Modeling & Validation` — 14 cells, all passed. BUT THIS WAS THE WRONG NOTEBOOK. It did W1/W2's job (PolyRidge/GP regression, model selection, 2028 forecasts). Results exist but are NOT Anas's W4 deliverable. GP locked at 0.152 NRMSE, zero physical violations, 2028 fans exported.

## ANAS'S ACTUAL W4 ROLE (corrected understanding)
Anas is the TIMELINE MECHANIC and REFEREE. His job is:
1. Build the backtest engine (the ruthless evaluation scoring script)
2. Fix the short-horizon time bug in temporal_utils.py
3. Build his own temporal challenger: strict b=0 exponential decline with physical caps
4. NOT build PolyRidge/GP models (that's Nazrul W1 and Afiq W2)

## BRAIN'S CELL 18 DESIGN (already produced)
You designed a 10-cell "03 Timeline Mechanics EDA" notebook focused on:
- Analysis 1: Matched free-b vs strict b=0 ablation (Cells 4-6)
- Analysis 2: Quarter-by-quarter error trajectory (Cells 7-8)
- Analysis 3: Physical water bounds diagnosis (Cells 9-10)
This is the CORRECT next notebook for Anas.

## COMPETITION DELIVERABLES REQUIRED
1. Source code + model files (zipped as {team_name}.zip)
2. PowerPoint documentation
3. Power BI dashboard (from 09 Template Deliverable.xlsx)
4. 7-minute YouTube video

## TEAM ARCHITECTURE
| Member | Workstream | Role |
|--------|-----------|------|
| Nazrul | W1: Polynomial Ridge | Stable baseline + standard decline → 2028 |
| Afiq | W2: ARD GP | Adaptive smooth mapping + uncertainty → 2028 |
| Hazeem | W3: Anchored decline decoder | Builds reference temporal decoder |
| Anas | W4: Alternative temporal + validation | Backtester, time-series bug fixes, strict b=0 challenger |

Now answer:

1. **Immediate next step**: Should Anas build the Timeline Mechanics EDA (from your cell 18 design) as the correct notebook 03? Or should he rename/repurpose what exists?

2. **Priority ordering**: What should Anas do in what order? Consider the competition deadline.

3. **The wrong notebook 03**: What do we do with the PolyRidge/GP notebook that already exists? Delete it? Rename it as a team-shared resource? Keep it as-is?

4. **Temporal_utils.py bug fix**: The short-horizon forecast() failure — should this be fixed BEFORE or AFTER the Timeline Mechanics EDA?

5. **End-state vision**: What does Anas's COMPLETED workstream look like? What are the exact deliverables (notebooks, files, tables, figures) that Anas personally owns?

6. **Team integration timing**: When should Anas's outputs feed into the team's combined deliverable? What handoff format?

7. **What can be dropped?** Given limited time, what from the Brain's cell 18 design can be simplified or skipped without hurting the competition submission?

Be specific. Give me a numbered action plan with estimated effort for each step.
""", follow_up=True)

In [0]:
answer = ask_brain("""
Give me a FULL, DETAILED interpretation of the notebook 04 Strict Timeline & Referee Validation results. This is Anas's W4 final deliverable.

Here are the exact results from all 8 cells that ran successfully:

## Cell 3 — Config
- temporal_utils.py hash: 3ba3b98c (clean)
- backtest_engine.py hash: 13d59280 (new referee engine)
- Water rate cap: 15,853 STB/day (2.0x observed 2008)
- Dev cases: 85, Train: 70, Val: 15

## Cell 4 — Strict b=0 Challenger Model
- Case 1 test: D_oil=0.1367/yr, GOR=0.3634
- Oil 2028: 46.7M STB, Gas: 17.0M MSCF, Water: 87.9M STB
- Water capped at 15,853 STB/day

## Cell 5 — Backtest Predictions
- Total predictions: 14,280 (exact match to expected: 85 × 4 experiments × 56 quarters × 3 phases)
- Fit records: 340 (85 cases × 4 experiments)
- Zero cases skipped

## Cell 6 — Referee Engine Scoring
- Validation: PASSED
- MACRO SCORES:
  - Strict_b0_Anas: mean_dev_NRMSE = 0.152523, worst_exp_NRMSE = 0.300573, median_dev_NRMSE = 0.133147

- PHASE SCORES by experiment (train / val increment_NRMSE):
  - 2003→3yr: oil 0.2105/0.2187, gas 0.2103/0.2185, water 0.2072/0.1945
  - 2003→5yr: oil 0.3013/0.3164, gas 0.3010/0.3161, water 0.2929/0.2760
  - 2004→3yr: oil 0.0532/0.0630, gas 0.0530/0.0624, water 0.0525/0.0539
  - 2005→3yr: oil 0.0335/0.0317, gas 0.0335/0.0316, water 0.0656/0.0632

- PHYSICAL VIOLATIONS:
  - Train: 0 negative cumulative, 0 decreasing steps
  - Validation: 0 negative cumulative, 0 decreasing steps

- VERDICT: PASS | mean_dev=0.1525 | worst_exp=0.3006 | violations=0

## Cell 7 — 2028 Forecasts (Val cases 71-85)
- 1,200 forecast rows (15 cases × 80 quarterly dates)
- 2028 endpoint statistics:
  - oil_cum: mean=50.6M, range=[39.3M, 60.1M]
  - gas_cum: mean=18.4M, range=[14.3M, 21.8M]
  - water_cum: mean=93.7M, range=[73.4M, 112.3M]

## Cell 8 — W4 Handoff
- w4_manifest.json + W4_HANDOFF.md exported
- Prediction contract defined for team

## Context from earlier notebooks:
- Notebook 90 (wrong nb, exploratory): GP 0.152122 NRMSE, PolyRidge 0.152544 (effectively tied)
- Notebook 02 (Temporal EDA): Separate decoder 0.155 NRMSE, Constant 0.276
- Notebook 03 (Timeline Mechanics EDA): Free-b vs b=0 divergence 29.5%, b=0 wins 66.7% of pairs
- Dataset: 100 cases (70 train, 15 val, 15 test), 4 uncertainty params, porosity multiplier dominant

Please interpret:
1. How does the strict b=0 challenger (0.1525 mean_dev) compare to the GP (0.152) and PolyRidge (0.153) from notebook 90? What does this mean for Anas's W4 position?
2. Why does the 2003→5yr experiment have the highest error (0.30)? Is this expected?
3. Why are 2004→3yr and 2005→3yr so much better (0.03-0.06)? What does this say about the temporal dynamics?
4. The strict b=0 model has ZERO physical violations vs the GP/PolyRidge which also had zero. How significant is this?
5. The water cumulative range (73-112M STB) vs oil (39-60M STB) — what does this say about the reservoir?
6. Is the 2x water cap (15,853 STB/day) appropriate? Could it be too loose or too tight?
7. How do these results position Anas's W4 workstream relative to W1 (Nazrul) and W2 (Afiq)?
8. Any concerns or weaknesses the team should know about?
9. What would you recommend for the competition PowerPoint and video based on these results?
10. Final verdict: Is W4 complete and ready for team integration?
""", follow_up=True)